In [1]:
from langchain_community.utilities import SQLDatabase
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
db_user=os.getenv("DB_USER")
db_password=os.getenv("DB_PASSWORD")
db_host=os.getenv("DB_HOST")
db_name=os.getenv("DB_NAME")
db=SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")
print(db.dialect)

mysql


In [3]:
db.get_usable_table_names()

['signup', 'signuptwo']

In [4]:
db.run("SELECT * FROM signup LIMIT 5;")

"[('0', 'Aryan Raina', 'Kamal Raina', datetime.datetime(2003, 2, 4, 0, 0), 'Male', 'aryanraina2021@gmail.com', 'Unmarried', 'Lane no.22 Block no.130 Flat no.18 Jagti', 'Jammu', '181221', 'Jammu and kashmir'), ('0', 'Rishika Raina', 'Kamal Raina', datetime.datetime(2005, 7, 27, 0, 0), 'Female', 'rishikaraina2005@gmail.com', 'Unmarried', 'Lane no.22 block no.130 Flat no.18 Jagti', 'Jammu', '181221', 'J&K'), ('6571', 'Kamal Raina', 'Janki Nath Raina', datetime.datetime(2024, 7, 9, 0, 0), 'Male', 'kamalraina1969@gmail.com', 'married', 'ad ', 'dddd', '181221', 'dinin'), ('4374', 'rajan', 'bhan', datetime.datetime(2017, 7, 4, 0, 0), 'Male', 'rajan44@gmail.com', 'married', 'Deathvalley ', 'Secret', '9000000', 'Unknown'), ('5015', 'rajan bhan', 'rajat bhan', datetime.datetime(2000, 7, 1, 0, 0), 'Male', 'rajanbhan420@gmail.com', 'married', 'deathvalley', 'unknown', '910200', 'unknown')]"

In [5]:
from langchain_community.chat_models import ChatOllama
llm = ChatOllama(model="llama3", temperature=0)

In [7]:
llm.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3', 'created_at': '2025-04-27T15:28:05.4008568Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 4315004300, 'load_duration': 82046600, 'prompt_eval_count': 17, 'prompt_eval_duration': 1709446200, 'eval_count': 8, 'eval_duration': 2521312300}, id='run-b9883c53-73ad-4171-bc8b-1f263c4d4704-0')

In [8]:
db.run("SELECT * FROM signup LIMIT 5;")

"[('0', 'Aryan Raina', 'Kamal Raina', datetime.datetime(2003, 2, 4, 0, 0), 'Male', 'aryanraina2021@gmail.com', 'Unmarried', 'Lane no.22 Block no.130 Flat no.18 Jagti', 'Jammu', '181221', 'Jammu and kashmir'), ('0', 'Rishika Raina', 'Kamal Raina', datetime.datetime(2005, 7, 27, 0, 0), 'Female', 'rishikaraina2005@gmail.com', 'Unmarried', 'Lane no.22 block no.130 Flat no.18 Jagti', 'Jammu', '181221', 'J&K'), ('6571', 'Kamal Raina', 'Janki Nath Raina', datetime.datetime(2024, 7, 9, 0, 0), 'Male', 'kamalraina1969@gmail.com', 'married', 'ad ', 'dddd', '181221', 'dinin'), ('4374', 'rajan', 'bhan', datetime.datetime(2017, 7, 4, 0, 0), 'Male', 'rajan44@gmail.com', 'married', 'Deathvalley ', 'Secret', '9000000', 'Unknown'), ('5015', 'rajan bhan', 'rajat bhan', datetime.datetime(2000, 7, 1, 0, 0), 'Male', 'rajanbhan420@gmail.com', 'married', 'deathvalley', 'unknown', '910200', 'unknown')]"

In [9]:
from typing_extensions import TypedDict


class State(TypedDict):
    question: str
    query: str
    result: str
    answer: str

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.runnables import Runnable

response_schemas = [
    ResponseSchema(name="sql_query", description="The SQL query to answer the user's question.")
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = parser.get_format_instructions()

prompt = PromptTemplate(
    template="""================================ System Message ================================

Given an input question, create a syntactically correct {dialect} query to run to help find the answer. Unless the user specifies in his question a specific number of rows they wish to obtain, always limit your query to at most {top_k} results. You can order the results by a relevant column to return the most interesting examples in the database.

Never query for all the columns from a specific table, only ask for the few relevant columns given the question.

Pay attention to use only the column names that you can see in the schema description. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

Only use the following tables:
{table_info}

{format_instructions}
=============================== Human Message =================================

Question: {input}""",
    input_variables=["input", "dialect", "table_info", "top_k"],
    partial_variables={"format_instructions": format_instructions}
)


# Build the chain only once
chain: Runnable = prompt | llm | parser

def write_query(state: State) -> State:
    question = state["question"]
    if not question.strip().endswith("?") :
        question = question.strip() + "?"
    response = chain.invoke({
        "dialect": db.dialect,
        "top_k": 5,
        "table_info": db.get_table_info(),
        "input": question,
    })
    if "sql_query" not in response or not response["sql_query"]:
        raise ValueError("The response does not contain a valid SQL query.")
    return {"query": response["sql_query"]}


In [11]:
write_query({"question": "show me the father name of Kamal Raina"})

{'query': "SELECT father_name FROM signup WHERE formno = '6571' LIMIT 1"}

In [12]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
def execute_query(state: State):
    execute_query_tool=QuerySQLDataBaseTool(db=db)
    return {"result": execute_query_tool.invoke(state["query"])}

In [13]:
execute_query({"query": "SELECT father_name FROM signup WHERE formno = '6571'"})

{'result': "[('Janki Nath Raina',)]"}

In [22]:
response = write_query({"question": "show me the father name of Kamal Raina ?"})
response.get("query","")

"SELECT father_name FROM signup WHERE formno = '6571' OR (formno = '0' AND father_name = 'Kamal Raina')"

In [24]:
response= write_query({"question": "show me all rows where there is jammu ?"})
sql = response.get("query", "")
print("Final SQL query:", sql)


if sql:
    try:
        result = execute_query(response)
        print("Query Result:", result)
    except Exception as e:
        print(f"Error executing query: {e}")
else:
    print("No valid SQL query generated.")

Final SQL query: SELECT signup.formno, signup.name FROM signup WHERE signup.city = 'Jammu' OR signup.state = 'Jammu and kashmir' LIMIT 5
Query Result: {'result': "[('0', 'Aryan Raina'), ('0', 'Rishika Raina')]"}


In [20]:
def generate_answer(state: State):
    prompt = (
        "Given the following user question, corresponding SQL query, "
        "and SQL result, answer the user question in simple language as shown in sql result.\n\n"
        f'Question: {state["question"]}\n'
        f'SQL Query: {state["query"]}\n'
        f'SQL Result: {state["result"]}'
    )
    response = llm.invoke(prompt)
    return {"answer": response.content}

In [ ]:
generate_answer({"question":"show me all rows where address is jammu ?","query":"SELECT father_name FROM signup WHERE formno = '6571'",'result': "[('Janki Nath Raina',)]"})

{'answer': "The father's name of Kamal Raina is Janki Nath Raina."}

In [15]:
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence(
    [write_query, execute_query, generate_answer]
)
graph_builder.add_edge(START, "write_query")
graph = graph_builder.compile()

In [21]:
all_steps= []
for step in graph.stream(
    {"question": "give me all the rows of signup table?"}, stream_mode="updates"
):
    all_steps.append(step)

In [22]:
all_steps[2]

{'generate_answer': {'answer': 'The user question is: "Give me all the rows of signup table?"\n\nTo answer this, we can modify the SQL query to remove the LIMIT clause, which specifies the number of rows to return. Here\'s the modified query:\n\nSELECT formno, name, father_name, dob, gender FROM signup;\n\nThis will return all the rows from the signup table.\n\nNote that the original query was limited to 5 rows with `LIMIT 5`, but by removing this clause, we can retrieve all the rows in the table.'}}

In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
sql_agent_toolkit = SQLDatabaseToolkit(db=db,llm=llm)
tools = sql_agent_toolkit.get_tools()
tools


NameError: name 'db' is not defined

In [52]:

from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

tools = toolkit.get_tools()

tools

[QuerySQLDataBaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002C720FEDCD0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002C720FEDCD0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002C720FEDCD0>),
 QuerySQLCheckerTool(description='Use this tool to 

In [53]:
from langchain import hub

prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")

assert len(prompt_template.messages) == 1
prompt_template.messages[0].pretty_print()

c:\Users\Aryan raina\OneDrive\Documents\LLM\sqlassistor\venv\Lib\site-packages\langsmith\client.py:271: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


================================ System Message ================================

You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to th

In [60]:
system_message = prompt_template.format(dialect=db.dialect, top_k=10)


In [61]:
system_message

'System: You are an agent designed to interact with a SQL database.\nGiven an input question, create a syntactically correct mysql query to run, then look at the results of the query and return the answer.\nUnless the user specifies a specific number of examples they wish to obtain, always limit your query to at most 10 results.\nYou can order the results by a relevant column to return the most interesting examples in the database.\nNever query for all the columns from a specific table, only ask for the relevant columns given the question.\nYou have access to tools for interacting with the database.\nOnly use the below tools. Only use the information returned by the below tools to construct your final answer.\nYou MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.\n\nDO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.\n\nTo start you should ALWAYS look at the tables in the datab

In [62]:
from langchain_core.messages import HumanMessage

from langchain.agents import initialize_agent
agent_executor = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)


In [64]:
agent_executor.run("show me the date of births of rishika raina and kamal raina")




> Entering new AgentExecutor chain...
Thought: To find the date of birth for Rishika Raina and Kamal Raina, I need to query a table that contains this information. Let's start by listing all tables in the database.

Action: sql_db_list_tables
Action Input: empty string
Observation: signup, signuptwo
Thought:Thought: Now that I have the list of tables, I can try to find the one that contains the date of birth information for Rishika Raina and Kamal Raina. Let me check the schema of these two tables.

Action: sql_db_schema
Action Input: signup, signuptwo
Observation: 
CREATE TABLE signup (
	formno VARCHAR(20), 
	name VARCHAR(20), 
	father_name VARCHAR(20), 
	dob DATETIME, 
	gender VARCHAR(20), 
	email VARCHAR(40), 
	marital_status VARCHAR(20), 
	address VARCHAR(40), 
	city VARCHAR(20), 
	pincode VARCHAR(20), 
	state VARCHAR(20)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_general_ci

/*
3 rows from signup table:
formno	name	father_name	dob	gender	email	marital_status	address	

"The date of birth for Rishika Raina is 2005-07-27 and Kamal Raina's date of birth is not available in these two tables."